# Grad-CAM comparison of trained CNNs

This notebook uses the class-weighted, oversampling, and augmentation models from the existing training notebooks. Their model variable is named `model`; all use 224×224 grayscale inputs, a `Rescaling(1/255)` layer, and a two-class softmax output. The confirmed mapping is `NORMAL=0`, `PNEUMONIA=1`.

**Checkpoint status:** I found no saved model/checkpoint files in the project. A model in another notebook's live kernel cannot be loaded here. Save it there after training with `model.save("baseline_cnn_class_weighted.keras")` (use corresponding filenames for the other two), then set the paths below. The training notebooks themselves are unchanged.

In [ ]:
# Setup: change the dataset or checkpoint paths here.
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

PROJECT_DIR = Path.cwd()  # Change if needed.
DATA_DIR = Path.home() / '.cache' / 'kagglehub' / 'datasets' / 'paultimothymooney' / 'chest-xray-pneumonia' / 'versions' / 'chest_xray_cnn_splits'
VALIDATION_DIR = DATA_DIR / 'val'
OUTPUT_DIR = PROJECT_DIR / 'gradcam_analysis_outputs'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

# No checkpoints were found. After saving a model from its training notebook,
# replace None with that .keras file's path.
CLASS_WEIGHTED_PATH = None  # e.g. PROJECT_DIR / 'baseline_cnn_class_weighted.keras'
OVERSAMPLING_PATH = None    # e.g. PROJECT_DIR / 'baseline_cnn_oversampling.keras'
AUGMENTATION_PATH = None    # e.g. PROJECT_DIR / 'baseline_cnn_augmentation.keras'

CHECKPOINTS = {
    'class_weighted': CLASS_WEIGHTED_PATH,
    'oversampling': OVERSAMPLING_PATH,
    'augmentation': AUGMENTATION_PATH,
}
print('Validation folder:', VALIDATION_DIR)
print('TensorFlow:', tf.__version__)

## Load the existing validation set and saved models

The validation loader matches the training notebooks: grayscale, resized to 224×224, integer labels, and `shuffle=False`. Validation images are not augmented or otherwise changed. Model files must be full `.keras` models saved from the trained `model` variable; weights-only files are not enough to reconstruct these notebook-defined CNNs without duplicating their architecture.

In [ ]:
validation_ds = tf.keras.utils.image_dataset_from_directory(
    VALIDATION_DIR,
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False,
)
images = np.concatenate([x.numpy() for x, _ in validation_ds])
labels = np.concatenate([y.numpy() for _, y in validation_ds])
if images.shape[1:] != (*IMAGE_SIZE, 1):
    raise ValueError(f'Unexpected validation image shape: {images.shape}')
print('Class mapping:', {name: index for index, name in enumerate(CLASS_NAMES)})
print('Validation counts:', dict(zip(CLASS_NAMES, np.bincount(labels, minlength=2))))

models = {}
for name, path in CHECKPOINTS.items():
    if path is None:
        print(f'{name}: unavailable; save the trained model and set its path above.')
        continue
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT_DIR / path
    if not path.is_file():
        print(f'{name}: checkpoint not found: {path}')
        continue
    model = tf.keras.models.load_model(path, compile=False, safe_mode=True)
    if model.input_shape != (None, 224, 224, 1) or model.output_shape != (None, 2):
        raise ValueError(f'{name}: checkpoint has incompatible input/output shapes.')
    models[name] = model
    print(f'{name}: loaded {path}')

if not models:
    print('No trained checkpoints loaded. Save them from the training notebooks to generate Grad-CAMs.')

## Grad-CAM

The target is the last `Conv2D` layer, which is the final spatial feature layer in the existing CNNs, before pooling and the classifier.

In [ ]:
def gradcam(model, image, class_index):
    image = tf.convert_to_tensor(image[None, ...], dtype=tf.float32)
    model(image, training=False)  # Build the model if needed.
    last_conv = next(layer for layer in reversed(model.layers)
                     if isinstance(layer, tf.keras.layers.Conv2D))
    cam_model = tf.keras.Model(model.inputs, [last_conv.output, model.outputs[0]])
    with tf.GradientTape() as tape:
        features, scores = cam_model(image, training=False)
        score = scores[:, class_index]
    gradients = tape.gradient(score, features)
    weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heatmap = tf.nn.relu(tf.reduce_sum(features * weights, axis=-1))[0]
    heatmap = tf.image.resize(heatmap[..., None], IMAGE_SIZE)[..., 0]
    return (heatmap / (tf.reduce_max(heatmap) + 1e-8)).numpy()

## Compare the same validation X-rays

The first correctly and incorrectly classified validation examples for the first available model are used for comparison. Every loaded model is then shown on those same images, where possible. If there are no checkpoints, no predictions or figures are produced.

In [ ]:
if not models:
    print('Analysis not run: no model checkpoints are available.')
else:
    predictions = {
        name: model.predict(images, batch_size=BATCH_SIZE, verbose=0)
        for name, model in models.items()
    }
    reference_name = next(iter(models))
    reference_labels = predictions[reference_name].argmax(axis=1)
    correct = np.flatnonzero(reference_labels == labels)
    incorrect = np.flatnonzero(reference_labels != labels)
    example_indices = []
    if len(correct):
        example_indices.append(int(correct[0]))
    if len(incorrect):
        example_indices.append(int(incorrect[0]))
    if not example_indices:
        print('No validation examples were selected.')

    for index in example_indices:
        for name, model in models.items():
            scores = predictions[name][index]
            predicted = int(np.argmax(scores))
            confidence = float(scores[predicted])
            heatmap = gradcam(model, images[index], predicted)
            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(images[index, ..., 0], cmap='gray', vmin=0, vmax=255)
            axes[0].set_title('Original X-ray')
            axes[1].imshow(images[index, ..., 0], cmap='gray', vmin=0, vmax=255)
            axes[1].imshow(heatmap, cmap='jet', alpha=0.45, vmin=0, vmax=1)
            axes[1].set_title('Grad-CAM overlay')
            axes[2].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
            axes[2].set_title('Heatmap')
            for axis in axes:
                axis.axis('off')
            fig.suptitle(
                f'{name} | true: {CLASS_NAMES[int(labels[index])]} | '
                f'predicted: {CLASS_NAMES[predicted]} ({confidence:.3f}) | validation #{index}'
            )
            fig.tight_layout()
            OUTPUT_DIR.mkdir(exist_ok=True)
            fig.savefig(OUTPUT_DIR / f'{name}_validation_{index}.png', dpi=140)
            plt.show()
            plt.close(fig)
    print('Figures saved to:', OUTPUT_DIR)

## Visual review and limitations

Compare each overlay with the original X-ray. Note whether high activation appears over the lung/thoracic region or instead includes borders, corners, text, or markers. These are qualitative observations only: there are no lung masks here, and Grad-CAM cannot prove that an artefact caused a prediction or that the model reasons clinically.

The preparation notebook combines the original training and validation images, then creates a stratified 10% validation split (`random_state=42`), keeping the original test split separate. The model notebooks load 224×224 grayscale images, use `NORMAL=0` and `PNEUMONIA=1`, and rescale inputs by 1/255 inside the CNN. The split code does not show patient-level grouping, so related images across splits are a possible leakage risk. Dataset bias and label noise are additional limitations. This is an academic prototype, not a diagnostic tool. This analysis uses validation only; it does not read or tune on the test set.

### Save each trained model

In each training notebook, run this **after its training cell** while that notebook's kernel still contains the trained `model` variable:

```python
model.save("baseline_cnn_class_weighted.keras")
```

Use `baseline_cnn_oversampling.keras` and `baseline_cnn_augmentation.keras` in the other two notebooks. Save into the project directory (or note the full path), then set the matching checkpoint variable in the setup cell and rerun this notebook. A separate notebook cannot access another notebook's live variables.